In [1]:
from CLfitter import *
from joblib import Parallel, delayed
import numpy as np
import os
from lmfit import Model, Parameters

In [11]:
config = {
    # -- file -----------------------------------------------------------
    'path':               'twisted-CrSBr-2deg-eels-SI_004.dm4',
    'core_loss_index':    3,
    'lowloss':            False,

    # -- energy window  (E_min, E_edge_onset, E_max) in eV --------------
    'energy_window':      (520, 570, 670),

    # -- pooling --------------------------------------------------------
    'pool_radius':        2,
    'gaussian_kernel':    True,

    # -- clustering -----------------------------------------------------
    'n_clusters':         4,

    # -- background fitting (fixed parameters on MC replicas) -----------
    'n_mc_replicas':      10,    # Per run
    'replica_version':    'covariance',  # 'triangular' | 'covariance'

    # -- parallelization ------------------------------------------------
    'parallelize':        True,
    'n_jobs':             4,
    'n_runs':             4,

    # -- output ---------------------------------------------------------
    'folder_path':        'CrSBr_fixed_params/output',

    # -- white line fitting ---------------------------------------------
    'L3_guess':           577,
    'L2_guess':           586,
    'fit_window':         5,
    'integration_window': 1.5,
    'fitdata_path':       'CrSBr_fixed_params/fitdata',
}

# Create output directory if it doesn't exist
os.makedirs(config['folder_path'], exist_ok=True)
os.makedirs(config['fitdata_path'], exist_ok=True)

## Background prediction



In [ ]:
data_handler = DataHandler()
try:
     data_handler.read_dm4_SI(config['path'], core_loss_index=3, lowloss=False)
except:
     data_handler.read_dm4_SI(config['path'], core_loss_index=2, lowloss=False)

pooler = Pooler(data_handler.signal, data_handler.si_size)
signal =  pooler.pool_data(sqr_radius=2, gaussian_kernel=True)
signal[signal<1] = 1

i = config['energy_window']

range_mask = (data_handler.energy_axis > i[0]) & (data_handler.energy_axis < i[-1])  # Range for clustering shape [n_E_1]
energy_range = data_handler.energy_axis[range_mask] # shape [n_E_1]
signal_range = signal.copy()[range_mask,:]  # shape [n_E_1, n_y*n_x]

pre_edge_mask = (energy_range > i[0]) & (energy_range < i[1])  # Range for pre-edge shape [n_E_2]

# ==============================================================================
# Simple Least Squares Fit
# ==============================================================================

# Select pre-edge region
E_pre = energy_range[pre_edge_mask]                                      # [n_E_2]
I_pre = signal_range[pre_edge_mask, :]                                   # [n_E_2, n_pix]

eps = 1e-12
E_log = np.log(E_pre)
I_log = np.log(I_pre + eps)

X = np.vstack([np.ones_like(E_log), -E_log]).T                          # [n_E_2, 2]

#solve for coefficients using least squares
coeffs, *_ = np.linalg.lstsq(X, I_log, rcond=None)

logA = coeffs[0, :]                                                      # [n_pix]
r = coeffs[1, :]                                                         # [n_pix]
A = np.exp(logA)

E_full = energy_range[:, None]                                           # [n_E_1, 1]
Prediction_least_squares = A[None, :] * (E_full ** (-r[None, :]))        # [n_E_1, n_pix]
np.savez(f'{config["fitdata_path"]}/pred_least_squares', pred = Prediction_least_squares)
# ==============================================================================
# replace NN with LSq fit on MC replica sampling from clusterer
# ==============================================================================

clusterer = ClusterAnalyzer(signal_range)
clusterer.cluster_data(n_clusters = 4, pre_edge_mask=pre_edge_mask,)
clusterer.cholesky_decomp()

def train_background(ii,energy_range, clusterer, i):
    n_E, n_pix = signal_range.shape
    n_clusters = clusterer.clusters_mean.shape[1]
    
    background_predictions = np.zeros((10, *signal_range.shape))

    # Precompute masks (same for all replicas)
    pre_edge_mask = (energy_range > i[0]) & (energy_range < i[1])
    E_pre = energy_range[pre_edge_mask]
    E_log = np.log(E_pre)

    # Design matrix for power-law fit
    X = np.vstack([np.ones_like(E_log), -E_log]).T  # [n_E_pre, 2]
    
    for replica in range(10):

        # Sample MC replica in log-space
        mc_replica_log = np.zeros_like(clusterer.clusters_mean)  # [n_E, n_clusters]

        for cluster_id in range(n_clusters):
            z = np.random.randn(E_pre.shape[0])
            L = clusterer.triangular_matices[:, :, cluster_id]
            mc_replica_log[:, cluster_id] = (
                clusterer.clusters_mean[:, cluster_id] + L @ z
            )

        # Convert to linear space
        mc_replica = np.exp(mc_replica_log)  # [n_E, n_clusters]

        # Expand to pixel space
        # assumes you have labels: cluster index per pixel
        mc_signal = mc_replica[:, clusterer.clusters]  # [n_E, n_pix]

        # Fit power-law on pre-edge
        I_pre = mc_signal  # [n_E_pre, n_pix]
        I_log = np.log(I_pre + 1e-12)

        coeffs, *_ = np.linalg.lstsq(X, I_log, rcond=None)

        logA = coeffs[0, :]
        r = coeffs[1, :]
        A = np.exp(logA)

        # Build background over full range
        E_full = energy_range[:, None]
        background = A[None, :] * (E_full ** (-r[None, :]))

        background_predictions[replica] = background
        
    np.save(f'{config["fitdata_path"]}/pred_{ii}', pred = background_predictions)

# Parallelize over 100 runs, each containing 10 MC replicas
n_jobs = 100 
Parallel(n_jobs=n_jobs, verbose = 100)(
    delayed(train_background)(ii, energy_range, clusterer, i)
    for ii in range(0, 100)
)


## White Line Fitting Functions

Model the edge region using two Gaussians (for the L3 and L2 peaks) and two shifted arctangent steps (for the continuum states).

In [ ]:
from lmfit import Model, Parameters
def _two_gauss_two_arctan(x,
                        A1, mu1, sigma1,
                        A2, mu2, sigma2,
                        C1, x01, w1,
                        C2, x02, w2):
    """
    Model = 2 Gaussians + 2 shifted arctan steps).
    """
    g1 = A1 * np.exp(-(x - mu1) ** 2 / (2 * sigma1 ** 2))
    g2 = A2 * np.exp(-(x - mu2) ** 2 / (2 * sigma2 ** 2))
    ar1 = C1 * (np.arctan((x - x01) / (w1 + 1e-12)) + np.pi / 2.0)
    ar2 = C2 * (np.arctan((x - x02) / (w2 + 1e-12)) + np.pi / 2.0)
    return g1 + g2 + ar1 + ar2


def _fit_and_integrate_white_lines(energy_axis, spectrum, mu_guesses, fit_window=5.0, integration_window=1.5):
    """
    LMfit: fit spectrum with 2 Gaussians + 2 shifted arctans.
    Returns: area1, area2, mu1, mu2, (step1_height, step2_height)
    """
    energy = energy_axis
    mask_fit = (energy > min(mu_guesses) - fit_window) & (energy < max(mu_guesses) + fit_window)
    x = energy[mask_fit]
    y = spectrum[mask_fit]

    if len(x) < 5 or np.all(y <= 0):
        return np.nan, np.nan,np.nan, np.nan,np.nan, np.nan,np.nan, np.nan

    # Initial guesses
    A1_guess = max(y[(x > mu_guesses[0]-0.5) & (x < mu_guesses[0]+0.5)].max(), 1e-3)
    A2_guess = max(y[(x > mu_guesses[1]-0.5) & (x < mu_guesses[1]+0.5)].max(), 1e-3)
    C_guess = max((y[-1] - y[0]) * 0.3, 1e-3)

    model = Model(_two_gauss_two_arctan)
    params = Parameters()

    #sharpness
    params.add("A1", value=A1_guess, min=0)
    params.add("A2", value=A2_guess, min=0)

    params.add("C1", value=C_guess, min=1)
    params.add("C2", value=C_guess, min=1)
   
    #sharpness
    params.add("sigma1", value=0.7, min=0.1, max=2.0)
    params.add("sigma2", value=0.7, min=0.1, max=2.0)

    params.add("w1", value=0.5, min=0.01, max=2)

    params.add("delta_w", value=0.0, min=-0.5, max=0.5)  # allow ±0.5 eV difference
    params.add("w2", expr="w1+delta_w")

    #onsets
    params.add("mu1", value=mu_guesses[0], min=mu_guesses[0]-2, max=mu_guesses[0]+2)
    params.add("mu2", value=mu_guesses[1], min=mu_guesses[1]-2, max=mu_guesses[1]+2)

    params.add('deltax01', value=0, min=-1, max=1)
    params.add("x01", expr = 'mu1+deltax01')
    params.add('deltax02', value=0, min=-1, max=1)
    params.add("x02", expr = 'mu2+deltax02')
    
    try:
        result = model.fit(y, params, x=x)
        best = result.best_values
        fit_y = result.best_fit

        mu1, mu2 = best["mu1"], best["mu2"]
        C1, x01, w1 = best["C1"], best["x01"], best["w1"]
        C2, x02, w2 = best["C2"], best["x02"], best["w2"]
        A1, A2, sigma1, sigma2 = best['A1'],best['A2'], best['sigma2'], best['sigma2']

        return C1, C2, A1, A2, sigma1, sigma2, mu1, mu2

    except Exception:
        print('bonk')
        return np.nan, np.nan,np.nan, np.nan,np.nan, np.nan,np.nan, np.nan

## White Line Analysis: Single Power-Law Fit

Calculate white line intensity using a single power-law fit on the pre-edge region (without MC replica uncertainty). This provides a baseline comparison.

In [ ]:
import matplotlib.pyplot as plt
white_line_intensity = np.zeros(961)

for i, spectrum in enumerate((signal_range-Prediction_least_squares).T):
    C1, C2, A1, A2, sigma1, sigma2, mu1, mu2 = _fit_and_integrate_white_lines(energy_range, spectrum, (577,586))
    white_line_intensity[i] = (A1*sigma1-A2*sigma2)/(C1+C2)

fig, ax  = plt.subplots(1,1, figsize=(5,5))

im = ax.imshow(np.log(white_line_intensity).reshape(31,31))
ax.set_yticks([])
ax.set_xticks([])
ax.set_xlabel('Spatial $x$ (a.u.)')
ax.set_ylabel('Spatial $y$ (a.u.)')

fig.colorbar(im, label = 'Log White Line Intensity (a.u.)')

plt.savefig('fixed_param_fit.svg')

## White Line Analysis: fixed parametric fit on MC replicas

Calculate white line intensity straight on the MC replicas. This forgoes the ability to interpolate between the clusters that the NN has.

In [ ]:
def _fit_single_pixel(spectrum, energy_range, mu_guess):
    """
    Fit a single pixel spectrum using your 2Gauss+2Arctan model.
    Reuses mu_guess to stabilize convergence.
    Returns the white-line intensity and updated mu_guess.
    """
    C1, C2, A1, A2, sigma1, sigma2, mu1, mu2 = _fit_and_integrate_white_lines(
        energy_range, spectrum, mu_guess
    )

    wl_intensity = (A1 * sigma1 - A2 * sigma2) / (C1 + C2) if not np.isnan(mu1) else np.nan

    # Update guess for next pixel
    if not np.isnan(mu1):
        mu_guess = (mu1, mu2)

    return wl_intensity, mu_guess


def _process_replica(signal_bg, energy_range):
    """
    Process one replica (all pixels) sequentially with parameter reuse.
    """
    n_pix = signal_bg.shape[1]
    wl_replica = np.zeros(n_pix)
    mu_guess = (577, 586)  # initial guess for first pixel

    for i in range(n_pix):
        spectrum = signal_bg[:, i]
        wl_replica[i], mu_guess = _fit_single_pixel(spectrum, energy_range, mu_guess)

    return wl_replica


def _process_file(file_idx, signal_range):
    """
    Process one file:
      - Load predictions
      - Subtract background
      - Fit all replicas
      - Save WL results
    """
    predictions = np.load(f'realresults_fixed_param/pred_{file_idx}.npz')['pred']
    n_replicas = len(predictions)
    n_pix = signal_range.shape[1]

    WL = np.zeros((n_replicas, n_pix))

    for xx, prediction in enumerate(predictions):
        # Background subtraction
        signal_bg = signal_range - prediction
        signal_bg[signal_bg < 0] = 0  # optional clipping

        # Fit all pixels in this replica sequentially with parameter reuse
        WL[xx] = _process_replica(signal_bg, energy_range)

    # Save results
    np.save(f'WL_results_fixed_params/results_{file_idx}.np', WL)
    return f"File {file_idx} done"

n_files = 100  # your 100 files
WL_results = Parallel(n_jobs=64)(
    delayed(_process_file)(ii, signal_range) for ii in range(n_files)
)

## Results Visualization

Visualize the white line intensity maps and uncertainty estimates from the MC ensemble.

In [ ]:
import numpy as np
import os
from glob import glob

# folder containing WL result files
folder = "WL_results_fixed_params"
file_pattern = os.path.join(folder, "results_*.np.npy")

# collect WL arrays from files that exist
wl_list = []

for file_path in glob(file_pattern):
    try:
        wl = np.load(file_path)  # shape [n_replicas, n_pix]
        wl_list.append(wl)
    except Exception:
        print(f"Skipping missing or corrupted file: {file_path}")

# combine all WL arrays along replica axis
# flatten: shape [n_files * n_replicas, n_pix]
all_wl = np.vstack(wl_list)  # shape [n_total_replicas, n_pix]

# compute median per pixel
wl_median = np.median(all_wl, axis=0)
lower, upper = np.percentile(all_wl, [25, 75], axis=0)

fig, ax = plt.subplots(3,1, figsize=(5,15))

ax[0].imshow(lower)
ax[1].imshow(wl_median)
ax[2].imshow(upper)

ax[0].set_title('25th Percentile')
ax[1].set_title('Median')
ax[2].set_title('75th Percentile')
for a in ax:
    a.set_yticks([])
    a.set_xticks([])
    a.set_xlabel('Spatial $x$ (a.u.)')
    a.set_ylabel('Spatial $y$ (a.u.)')
